# MGMT298D: Science and Strategy of AI## Week 8: LLM APIs — Prompt Engineering & RAG### UCLA Anderson School of Management

## OverviewIn this notebook we explore two approaches to building useful AI applications with LLMs:1. **Prompt Engineering** — crafting better prompts to improve the quality of the model's answers using only its built-in knowledge2. **Retrieval-Augmented Generation (RAG)** — giving the model access to external documents so it can answer questions about information it was never trained onWe compare both approaches on the same task: **answering employee questions about a fictional company's internal documents** — policies, product specs, and memos that the LLM has never seen and cannot possibly know.**API:** Google Gemini (free tier)

## Part 1: Setup & First API Call

### 1. Install & Import

In [ ]:
# Install required packages!pip install -q google-generativeai chromadb sentence-transformersimport google.generativeai as genaiimport numpy as npimport pandas as pdimport jsonimport textwrapfrom IPython.display import Markdown, display# Helper to print wrapped text nicelydef show(text, width=90):    print(textwrap.fill(text, width=width))

### 2. Configure Gemini APIGet a free API key from [Google AI Studio](https://aistudio.google.com/app/apikey) — no credit card required.

In [ ]:
# Paste your Gemini API key hereGEMINI_API_KEY = "YOUR_API_KEY_HERE"  # ← Replace with your keygenai.configure(api_key=GEMINI_API_KEY)# Initialize Gemini Flash (free tier)model = genai.GenerativeModel('gemini-2.0-flash')# Test with a simple callresponse = model.generate_content("Say hello in three languages.")print(response.text)

### 3. Understanding API ParametersTemperature controls randomness: lower = more focused, higher = more creative.

In [ ]:
# Same prompt, different temperaturesprompt = "Give me one creative name for a robot dog."print("Temperature comparison:\n")for temp in [0.0, 0.5, 1.0, 1.5]:    responses = []    for _ in range(3):        config = genai.GenerationConfig(temperature=temp, max_output_tokens=30)        r = model.generate_content(prompt, generation_config=config)        responses.append(r.text.strip())    print(f"  T={temp}: {responses}")

## Part 2: The Company — Athena Robotics

### 4. Load Company DocumentsBelow are the internal documents for **Athena Robotics**, a fictional robotics startup.These documents have never appeared on the internet — the LLM cannot know this information.> **Instructors:** Replace or modify these documents with your own content.> The rest of the notebook will work with any text you put here.

In [ ]:
# ============================================================# FICTIONAL COMPANY DOCUMENTS — Edit these freely# ============================================================company_docs = {    "employee_handbook": """ATHENA ROBOTICS — EMPLOYEE HANDBOOK (2025 Edition)COMPANY OVERVIEWAthena Robotics was founded in 2019 by CEO Priya Chandrasekaran and CTO Marcus Webbin Oakland, California. The company develops autonomous warehouse robots for mid-sizee-commerce fulfillment centers. As of January 2025, Athena has 340 employees acrossthree offices: Oakland (HQ, 210 employees), Austin (85 employees), and Toronto (45 employees).PAID TIME OFFAll full-time employees receive 22 days of PTO per year, accrued monthly at 1.83 days/month.PTO does not roll over — unused days expire on December 31. Employees in their first yearreceive a prorated amount starting from their hire date. Part-time employees (20+ hrs/week)receive 11 days. PTO requests must be submitted at least 10 business days in advance throughthe Athena HR Portal. Requests for more than 5 consecutive days require manager AND VP approval.REMOTE WORK POLICYAthena operates on a hybrid schedule: employees are required in-office Tuesday, Wednesday,and Thursday. Monday and Friday are optional remote days. Fully remote arrangements areavailable only for roles explicitly marked "remote-eligible" in the job posting and requireVP-level approval. Remote employees receive a one-time $1,500 home office stipend.All employees must be based within 50 miles of an Athena office unless granted a locationexception by the Chief People Officer.PARENTAL LEAVEPrimary caregivers receive 16 weeks of fully paid parental leave. Secondary caregiversreceive 8 weeks fully paid. Parental leave can be taken in two non-consecutive blockswithin 12 months of the child's birth or adoption date. Employees must have been withAthena for at least 6 months to be eligible.EXPENSE POLICYMeals during business travel: up to $75/day ($25 breakfast, $25 lunch, $25 dinner).Client entertainment: up to $150/person with pre-approval from Finance.Software subscriptions: up to $50/month without approval; $50-$500/month requiresmanager approval; over $500/month requires VP approval.All expenses must be submitted within 30 days via Concur with itemized receipts.Late submissions (31-60 days) require manager approval. Submissions after 60 dayswill not be reimbursed.PERFORMANCE REVIEWSReviews are conducted semi-annually in March and September. Employees are rated on a5-point scale: Exceptional (5), Exceeds Expectations (4), Meets Expectations (3),Needs Improvement (2), Unsatisfactory (1). Ratings of 1 or 2 trigger a PerformanceImprovement Plan (PIP) with a 90-day timeline. Annual bonuses are tied to ratings:Exceptional = 20% of base salary, Exceeds = 15%, Meets = 10%, Needs Improvement = 0%.""",    "product_spec_atlas": """PRODUCT SPECIFICATION: ATLAS 3.0 WAREHOUSE ROBOTOVERVIEWAtlas 3.0 is Athena Robotics' flagship autonomous mobile robot (AMR) designed forwarehouse pick-and-pack operations. It is the third generation of the Atlas platform,launched in September 2024.PHYSICAL SPECIFICATIONS- Dimensions: 24" x 20" x 36" (L x W x H)- Weight: 85 lbs (unloaded), max payload 60 lbs- Speed: up to 4.5 mph (loaded), 6.2 mph (unloaded)- Battery: 48V lithium iron phosphate, 8-hour runtime, 45-minute fast charge to 80%- Operating temperature: 32°F to 110°F- Noise level: < 55 dB at 3 feetNAVIGATION & SENSORS- Primary navigation: LiDAR (360° Velodyne VLP-16) + stereo camera array- Obstacle avoidance: ultrasonic sensors (8 units) + ToF depth sensors (4 units)- Localization: simultaneous localization and mapping (SLAM) with 2cm accuracy- Floor types: concrete, epoxy, rubber mat — NOT rated for outdoor or gravel surfacesSOFTWARE & INTEGRATION- Fleet management via Athena Command Center (cloud dashboard)- API: RESTful JSON API for WMS integration (supports SAP, Oracle, Manhattan Associates)- Max fleet size per facility: 200 robots with single Command Center instance- OTA firmware updates pushed weekly (maintenance window: Sunday 2-4 AM local time)- Pick accuracy: 99.7% (verified by independent audit, Q3 2024)PRICING- Hardware: $28,500 per unit- Annual software license: $4,200 per robot (includes Command Center access)- Volume discounts: 10+ units = 8% off hardware; 50+ units = 15% off hardware- Installation & mapping: $3,500 per facility (one-time)- Maintenance contract: $2,400/year per robot (covers parts and on-site repair)WARRANTYStandard warranty: 2 years parts and labor from delivery date. Battery warranty:3 years or 5,000 charge cycles, whichever comes first. Warranty void if robot isoperated outside specified temperature range or on non-approved floor surfaces.""",    "product_spec_hermes": """PRODUCT SPECIFICATION: HERMES DELIVERY DRONEOVERVIEWHermes is Athena Robotics' autonomous last-mile delivery drone, currently in betatesting with select partners. Expected general availability: Q3 2025.PHYSICAL SPECIFICATIONS- Wingspan: 48 inches, folded dimensions: 14" x 14" x 8"- Weight: 12.5 lbs (unloaded), max payload 8 lbs- Flight speed: 35 mph cruising, 45 mph max- Range: 12 miles round trip (with 5 lb payload)- Battery: 22.2V 6S LiPo, 25-minute flight time, 90-minute full charge- Operating conditions: wind up to 20 mph, light rain (IP54 rated), not rated for snowNAVIGATION & SAFETY- GPS + RTK for centimeter-level positioning- Downward-facing camera for precision landing- ADS-B receiver for manned aircraft detection- Automatic return-to-base on low battery (< 15%) or signal loss (> 30 seconds)- Parachute deployment system for emergency descent- Geo-fencing: operator-defined no-fly zones enforced via onboard firmwarePRICING (BETA PROGRAM)- Hardware: $8,900 per unit (beta pricing, expected to increase at GA)- Monthly software fee: $350/drone (includes fleet management + airspace integration)- Landing pad hardware: $1,200 per site- Beta program minimum: 5 drones per partnerREGULATORY STATUS- FAA Part 107 compliant for visual line of sight (VLOS) operations- Beyond Visual Line of Sight (BVLOS) waiver: approved in 3 states (Texas, Arizona, Nevada)- Pending BVLOS approval in California, Florida, Ohio (expected Q2 2025)""",    "internal_memo_q4_review": """INTERNAL MEMO — CONFIDENTIALTO: All HandsFROM: Priya Chandrasekaran, CEODATE: January 15, 2025RE: Q4 2024 Results & 2025 PrioritiesTeam,I'm excited to share our Q4 and full-year 2024 results.Q4 2024 HIGHLIGHTS- Revenue: $18.2M (up 34% vs Q4 2023)- Full-year 2024 revenue: $62.7M (up 41% YoY)- Atlas 3.0 units shipped in Q4: 847 (total installed base: 2,340 units)- Net new customers in Q4: 23 (total active customers: 156)- Gross margin: 52% (up from 47% in Q4 2023)- Customer NPS: 72 (industry average: 45)KEY WINS- Signed our largest deal ever: 120 Atlas units for FulfillCo ($4.1M TCV)- Hermes drone beta launched with 4 partners in Texas and Arizona- Opened Toronto office to support Canadian expansion- Atlas 3.0 pick accuracy independently verified at 99.7%CHALLENGES- Supply chain delays on LiDAR sensors pushed 62 Atlas deliveries into Q1 2025- Engineering turnover reached 18% (above our 12% target); exit interviews cite  compensation competitiveness as primary concern- Hermes BVLOS California approval delayed from Q4 2024 to Q2 20252025 PRIORITIES1. Hit $95M revenue target (52% growth)2. Launch Hermes for general availability by Q3 20253. Reduce engineering turnover to below 14% — we are implementing a mid-year   compensation review cycle and increasing RSU grants for senior engineers4. Expand into European market — Berlin office planned for Q3 20255. Achieve SOC 2 Type II certification by end of Q2 2025As always, please direct questions to your team leads or reach out to me directly.— Priya""",    "internal_memo_engineering": """INTERNAL MEMO — ENGINEERING TEAMTO: Engineering All-HandsFROM: Marcus Webb, CTODATE: February 3, 2025RE: Atlas 4.0 Development Timeline & Technical DecisionsTeam,Following our architecture review last week, here are the confirmed decisions forAtlas 4.0 development:TIMELINE- Design freeze: April 30, 2025- Prototype build: May-July 2025- Internal testing: August-September 2025- Beta with 5 customers: October 2025- General availability: Q1 2026KEY TECHNICAL CHANGES (vs Atlas 3.0)1. NAVIGATION: Replacing Velodyne VLP-16 LiDAR with Ouster OS1-64. The OS1-64   provides 64 channels vs 16, enabling better vertical resolution in dense rack   environments. Cost increase of $1,200 per unit offset by volume pricing agreement.2. COMPUTE: Upgrading from NVIDIA Jetson AGX Orin to Jetson Thor. This gives us   4x the AI inference performance, enabling on-robot anomaly detection and real-time   inventory counting via camera feed.3. BATTERY: Moving to silicon-anode cells from Enovix. Expected runtime increase   from 8 hours to 11 hours. Fast charge time decreases from 45 min to 30 min (to 80%).4. PAYLOAD: Target 75 lbs max payload (up from 60 lbs) via redesigned chassis.   Mechanical engineering team is evaluating carbon fiber composite frame.5. SOFTWARE: Atlas 4.0 will ship with Athena OS 2.0, featuring multi-robot   collaborative picking (2-3 robots coordinating on large orders) and predictive   maintenance alerts based on motor current analysis.STAFFINGWe are allocating 45 engineers to Atlas 4.0 (30 from existing Atlas team, 15 new hires).Firmware team is hiring 6 additional embedded engineers — please refer candidates.Budget for Atlas 4.0 development through GA: $8.2M (approved by board in January).Let me know if you have questions. Weekly syncs start Monday at 10 AM Pacific.— Marcus""",}print(f"Loaded {len(company_docs)} company documents:")for name, text in company_docs.items():    word_count = len(text.split())    print(f"  • {name}: {word_count} words")print(f"\nTotal: {sum(len(t.split()) for t in company_docs.values())} words")

### 5. Define Evaluation QuestionsThese questions have clear factual answers from the documents above.We'll use them to compare prompt-only vs. RAG performance.

In [ ]:
# Questions with ground-truth answers from the documentseval_questions = [    {        "question": "How many days of PTO do full-time employees get per year?",        "answer": "22 days per year, accrued monthly at 1.83 days/month.",        "source": "employee_handbook"    },    {        "question": "What is the maximum payload capacity of the Atlas 3.0?",        "answer": "60 lbs.",        "source": "product_spec_atlas"    },    {        "question": "What was Athena Robotics' full-year 2024 revenue?",        "answer": "$62.7M, up 41% year-over-year.",        "source": "internal_memo_q4_review"    },    {        "question": "What is the price per unit of the Atlas 3.0 robot?",        "answer": "$28,500 per unit, with volume discounts (8% off for 10+ units, 15% off for 50+ units).",        "source": "product_spec_atlas"    },    {        "question": "How long is parental leave for primary caregivers?",        "answer": "16 weeks fully paid. Must have been with Athena for at least 6 months.",        "source": "employee_handbook"    },    {        "question": "What LiDAR sensor is planned for the Atlas 4.0?",        "answer": "Ouster OS1-64, replacing the Velodyne VLP-16 used in Atlas 3.0.",        "source": "internal_memo_engineering"    },    {        "question": "When is the Hermes drone expected to be generally available?",        "answer": "Q3 2025.",        "source": "product_spec_hermes"    },    {        "question": "What was the engineering turnover rate, and what is the target?",        "answer": "18%, above the 12% target. Company is implementing mid-year comp reviews and increasing RSU grants.",        "source": "internal_memo_q4_review"    },    {        "question": "What is the maximum fleet size per facility for Atlas robots?",        "answer": "200 robots with a single Command Center instance.",        "source": "product_spec_atlas"    },    {        "question": "What is the development budget for Atlas 4.0?",        "answer": "$8.2M, approved by the board in January.",        "source": "internal_memo_engineering"    },]print(f"Prepared {len(eval_questions)} evaluation questions.")print(f"\nSample:")for q in eval_questions[:3]:    print(f"  Q: {q['question']}")    print(f"  A: {q['answer']}")    print()

## Part 3: Prompt Engineering (Without RAG)

### 6. Zero-Shot: Just AskFirst, we ask Gemini each question with no context at all.Since Athena Robotics is fictional, the model has no way to know the answers.

In [ ]:
# Zero-shot: ask questions with no contextdef ask_zero_shot(question):    response = model.generate_content(        question,        generation_config=genai.GenerationConfig(temperature=0.2, max_output_tokens=200)    )    return response.text.strip()print("=" * 80)print("ZERO-SHOT (No Context)")print("=" * 80)zero_shot_answers = []for item in eval_questions:    answer = ask_zero_shot(item["question"])    zero_shot_answers.append(answer)    print(f"\nQ: {item['question']}")    print(f"Expected: {item['answer']}")    print(f"Model:    {answer[:150]}")    print("-" * 40)

### 7. Prompt with System InstructionsWe can improve by telling the model who it is and how to behave —but it still doesn't have access to the actual documents.

In [ ]:
# System-instructed prompt: tell the model it's an Athena employee assistantdef ask_with_system_prompt(question):    system_prompt = """You are an internal Q&A assistant for Athena Robotics, a warehouserobotics company founded in 2019 in Oakland, California. Answer employee questionsaccurately and concisely. If you don't know the specific answer, say so honestlyrather than guessing."""    full_prompt = f"{system_prompt}\n\nEmployee question: {question}"    response = model.generate_content(        full_prompt,        generation_config=genai.GenerationConfig(temperature=0.2, max_output_tokens=200)    )    return response.text.strip()print("=" * 80)print("WITH SYSTEM PROMPT (Still No Documents)")print("=" * 80)system_prompt_answers = []for item in eval_questions:    answer = ask_with_system_prompt(item["question"])    system_prompt_answers.append(answer)    print(f"\nQ: {item['question']}")    print(f"Expected: {item['answer']}")    print(f"Model:    {answer[:150]}")    print("-" * 40)

### 8. Few-Shot PromptingWe give the model a few example Q&A pairs to teach it the format and level of detailwe expect. This improves the *style* of answers but still can't provide facts it doesn't know.

In [ ]:
# Few-shot: provide example Q&A pairsdef ask_few_shot(question):    few_shot_prompt = """You are an internal Q&A assistant for Athena Robotics.Answer employee questions accurately. Here are some example Q&A pairs:Q: Where is Athena Robotics headquartered?A: Oakland, California. We also have offices in Austin (85 employees) and Toronto (45 employees).Q: What is our flagship product?A: The Atlas 3.0, an autonomous mobile robot for warehouse pick-and-pack operations.Q: Who is the CEO?A: Priya Chandrasekaran, who co-founded Athena in 2019 with CTO Marcus Webb.Now answer this employee question. If you don't know the specific details, say so.Q: {question}A:"""    response = model.generate_content(        few_shot_prompt.format(question=question),        generation_config=genai.GenerationConfig(temperature=0.2, max_output_tokens=200)    )    return response.text.strip()print("=" * 80)print("FEW-SHOT PROMPTING (Still No Documents)")print("=" * 80)few_shot_answers = []for item in eval_questions:    answer = ask_few_shot(item["question"])    few_shot_answers.append(answer)    print(f"\nQ: {item['question']}")    print(f"Expected: {item['answer']}")    print(f"Model:    {answer[:150]}")    print("-" * 40)

**Key Takeaway:** Even with good prompting, the model cannot answer questions aboutinformation it was never trained on. It either hallucinates plausible-sounding detailsor admits it doesn't know. This is the fundamental limitation that RAG solves.

## Part 4: Retrieval-Augmented Generation (RAG)

### 9. Chunk the DocumentsRAG works by splitting documents into smaller chunks, storing them in a vector database,and retrieving only the most relevant chunks for each question.

In [ ]:
# Split documents into chunks for retrievaldef chunk_document(name, text, chunk_size=500, overlap=50):    """Split a document into overlapping word-based chunks."""    words = text.split()    chunks = []    start = 0    while start < len(words):        end = start + chunk_size        chunk_text = ' '.join(words[start:end])        chunks.append({            'text': chunk_text,            'source': name,            'chunk_id': f"{name}_chunk_{len(chunks)}"        })        start += chunk_size - overlap    return chunks# Chunk all documentsall_chunks = []for name, text in company_docs.items():    chunks = chunk_document(name, text)    all_chunks.append(chunks)    print(f"  {name}: {len(chunks)} chunks")    # Flattenall_chunks = [c for doc_chunks in all_chunks for c in doc_chunks]print(f"\nTotal chunks: {len(all_chunks)}")

### 10. Build Vector DatabaseWe embed each chunk using a sentence-transformer model, then store them in ChromaDBfor fast similarity search.

In [ ]:
# Initialize embedding model (runs locally, no API needed)embedder = SentenceTransformer('all-MiniLM-L6-v2')print(f"Embedding model loaded: all-MiniLM-L6-v2 (dimension: {embedder.get_sentence_embedding_dimension()})")# Create ChromaDB collectionchroma_client = chromadb.Client()collection = chroma_client.create_collection(    name="athena_docs",    metadata={"hnsw:space": "cosine"})# Embed and store all chunkstexts = [c['text'] for c in all_chunks]embeddings = embedder.encode(texts, show_progress_bar=True).tolist()collection.add(    documents=texts,    embeddings=embeddings,    ids=[c['chunk_id'] for c in all_chunks],    metadatas=[{'source': c['source']} for c in all_chunks])print(f"\nVector database built: {collection.count()} chunks indexed")

### 11. Retrieval FunctionGiven a question, we find the most relevant document chunks using cosine similarity.

In [ ]:
# Retrieve top-k relevant chunks for a questiondef retrieve(question, top_k=3):    """Find the most relevant document chunks for a question."""    query_embedding = embedder.encode([question]).tolist()        results = collection.query(        query_embeddings=query_embedding,        n_results=top_k    )        retrieved = []    for i in range(len(results['documents'][0])):        retrieved.append({            'text': results['documents'][0][i],            'source': results['metadatas'][0][i]['source'],            'distance': results['distances'][0][i]        })    return retrieved# Test retrieval on one questiontest_q = eval_questions[0]['question']results = retrieve(test_q)print(f"Question: {test_q}\n")for i, r in enumerate(results):    print(f"Chunk {i+1} (source: {r['source']}, distance: {r['distance']:.4f}):")    print(f"  {r['text'][:200]}...")    print()

### 12. RAG: Retrieve → Augment → GenerateNow we combine retrieval with generation: find relevant chunks, inject them into theprompt as context, and let the model answer based on the actual documents.

In [ ]:
# RAG pipeline: retrieve context, then generate answerdef ask_with_rag(question, top_k=3):    """Full RAG pipeline: retrieve relevant chunks and generate an answer."""    # Step 1: Retrieve    chunks = retrieve(question, top_k=top_k)    context = "\n\n---\n\n".join([c['text'] for c in chunks])    sources = list(set(c['source'] for c in chunks))        # Step 2: Augment prompt with retrieved context    rag_prompt = f"""You are an internal Q&A assistant for Athena Robotics.Answer the employee's question using ONLY the context provided below.Be specific — include exact numbers, dates, and details from the documents.If the context doesn't contain the answer, say "I don't have that information."CONTEXT:{context}QUESTION: {question}ANSWER:"""        # Step 3: Generate    response = model.generate_content(        rag_prompt,        generation_config=genai.GenerationConfig(temperature=0.1, max_output_tokens=300)    )        return response.text.strip(), sourcesprint("=" * 80)print("RAG — Retrieval-Augmented Generation")print("=" * 80)rag_answers = []for item in eval_questions:    answer, sources = ask_with_rag(item["question"])    rag_answers.append(answer)    print(f"\nQ: {item['question']}")    print(f"Expected: {item['answer']}")    print(f"RAG:      {answer[:200]}")    print(f"Sources:  {sources}")    print("-" * 40)

## Part 5: Comparison

### 13. Side-by-Side Results

In [ ]:
# Build comparison tablecomparison_data = []for i, item in enumerate(eval_questions):    comparison_data.append({        'Question': item['question'][:60] + '...' if len(item['question']) > 60 else item['question'],        'Expected': item['answer'][:60] + '...' if len(item['answer']) > 60 else item['answer'],        'Zero-Shot': zero_shot_answers[i][:60] + '...' if len(zero_shot_answers[i]) > 60 else zero_shot_answers[i],        'Few-Shot': few_shot_answers[i][:60] + '...' if len(few_shot_answers[i]) > 60 else few_shot_answers[i],        'RAG': rag_answers[i][:60] + '...' if len(rag_answers[i]) > 60 else rag_answers[i],    })comparison_df = pd.DataFrame(comparison_data)print("Side-by-Side Comparison (truncated):\n")print(comparison_df.to_string(index=False))

### 14. LLM-as-Judge EvaluationWe use Gemini itself to score each answer against the ground truth on a 1-5 scale.This is a common evaluation technique for LLM outputs.

In [ ]:
# Use Gemini to score answer quality (1-5)def score_answer(question, expected, predicted):    """Ask Gemini to score how well the predicted answer matches the expected answer."""    judge_prompt = f"""You are an impartial judge evaluating answer quality.Score the PREDICTED answer against the EXPECTED answer on a scale of 1-5:5 = Perfect: Contains the correct specific facts4 = Good: Mostly correct with minor omissions3 = Partial: Some correct info but missing key details2 = Poor: Vague or mostly incorrect1 = Wrong: Completely wrong, hallucinated, or "I don't know"QUESTION: {question}EXPECTED ANSWER: {expected}PREDICTED ANSWER: {predicted}Respond with ONLY a single number (1-5)."""        response = model.generate_content(        judge_prompt,        generation_config=genai.GenerationConfig(temperature=0.0, max_output_tokens=5)    )    try:        score = int(response.text.strip()[0])        return min(max(score, 1), 5)    except:        return 1# Score all approachesprint("Scoring answers with LLM-as-Judge...\n")methods = {    'Zero-Shot': zero_shot_answers,    'System Prompt': system_prompt_answers,    'Few-Shot': few_shot_answers,    'RAG': rag_answers}scores = {method: [] for method in methods}for i, item in enumerate(eval_questions):    for method, answers in methods.items():        score = score_answer(item['question'], item['answer'], answers[i])        scores[method].append(score)        # Build scores DataFramescores_df = pd.DataFrame(scores)scores_df.index = [q['question'][:50] + '...' for q in eval_questions]print(scores_df)print(f"\nMean Scores:")for method in methods:    print(f"  {method:15s}: {np.mean(scores[method]):.2f} / 5.00")

### 15. Visualization

In [ ]:
import matplotlib.pyplot as pltimport seaborn as snssns.set_style('whitegrid')# Bar chart of mean scoresfig, axes = plt.subplots(1, 2, figsize=(14, 5))# --- Mean Score by Method ---method_names = list(methods.keys())mean_scores = [np.mean(scores[m]) for m in method_names]colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']bars = axes[0].bar(method_names, mean_scores, color=colors, alpha=0.85, edgecolor='black', linewidth=0.8)for bar, score in zip(bars, mean_scores):    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,                 f'{score:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')axes[0].set_ylabel('Mean Score (1-5)', fontsize=12)axes[0].set_title('Answer Quality by Method', fontsize=14, fontweight='bold')axes[0].set_ylim([0, 5.5])axes[0].grid(True, alpha=0.3, axis='y')# Add divideraxes[0].axvline(x=2.5, color='gray', linestyle=':', linewidth=1.5, alpha=0.6)axes[0].text(1, 5.3, 'Prompt Engineering Only', ha='center', fontsize=9, fontstyle='italic', color='gray')axes[0].text(3, 5.3, 'With Retrieval', ha='center', fontsize=9, fontstyle='italic', color='gray')# --- Heatmap of individual scores ---heatmap_data = pd.DataFrame(scores)heatmap_data.index = [f"Q{i+1}" for i in range(len(eval_questions))]sns.heatmap(heatmap_data, annot=True, cmap='RdYlGn', vmin=1, vmax=5,             fmt='d', linewidths=0.5, ax=axes[1],            cbar_kws={'label': 'Score (1-5)'})axes[1].set_title('Scores by Question & Method', fontsize=14, fontweight='bold')axes[1].set_ylabel('Question')plt.tight_layout()plt.show()

### 16. Key Takeaways

In [ ]:
# Print summaryprint("=" * 70)print("SUMMARY: Prompt Engineering vs. RAG")print("=" * 70)pe_best = max(np.mean(scores['Zero-Shot']), np.mean(scores['System Prompt']), np.mean(scores['Few-Shot']))rag_score = np.mean(scores['RAG'])print(f"""Best Prompt Engineering Score: {pe_best:.2f} / 5.00RAG Score:                     {rag_score:.2f} / 5.00KEY INSIGHTS:1. PROMPT ENGINEERING is powerful for shaping HOW the model responds   (format, tone, level of detail) but cannot conjure facts it doesn't have.2. RAG solves the knowledge gap by retrieving relevant documents and   injecting them into the prompt as context.3. The two approaches are COMPLEMENTARY, not competing:   - Good prompts + good retrieval = best results   - RAG without good prompts can still produce poorly formatted answers   - Good prompts without RAG cannot answer questions about private data4. RAG is especially valuable for:   - Proprietary/internal company information   - Frequently updated data (policies, pricing, product specs)   - Reducing hallucination risk on factual questions""")